In [2]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('toronto_transportation.db')
print("Database connected. Running the KPI Engine...\n")

# KPI 1: Cancellations by Hour
query_cancellations = """
SELECT 
    SUBSTR(hr, 12, 2) AS hour_of_day,
    SUM(driver_cancelled_trips + passenger_cancelled_trips) AS total_cancellations
FROM 
    vehicle_operating_data
GROUP BY 
    hour_of_day
ORDER BY 
    hour_of_day ASC;
"""
df_cancellations = pd.read_sql_query(query_cancellations, conn)
df_cancellations.to_csv('kpi_1_cancellations_by_hour.csv', index=False)
print("KPI 1: Cancellations saved. Preview:")
display(df_cancellations.head(3))

# KPI 2: Daily Revenue & Idle Time
query_revenue = """
SELECT 
    SUBSTR(hr, 1, 10) AS trip_date,
    SUM(fares) AS total_revenue_cad,
    SUM(time_available) AS total_idle_minutes
FROM 
    vehicle_operating_data
GROUP BY 
    trip_date
ORDER BY 
    trip_date ASC;
"""
df_revenue = pd.read_sql_query(query_revenue, conn)
df_revenue.to_csv('kpi_2_revenue_and_idle.csv', index=False)
print("\nKPI 2: Revenue & Idle Time saved. Preview:")
display(df_revenue.head(3))

# KPI 3: Pooled Trips Efficiency by Date
query_pooled = """
SELECT 
    SUBSTR(hr, 1, 10) AS trip_date,
    SUM(pooled_trips_started) AS total_pooled_trips,
    SUM(reported_trips_started) AS total_trips,
    (CAST(SUM(pooled_trips_started) AS FLOAT) / NULLIF(SUM(reported_trips_started), 0)) * 100 AS pool_efficiency_pct
FROM 
    vehicle_operating_data
GROUP BY 
    trip_date
ORDER BY 
    trip_date ASC;
"""
df_pooled = pd.read_sql_query(query_pooled, conn)
df_pooled.to_csv('kpi_3_pooled_efficiency.csv', index=False)
print("\nKPI 3: Pooled Trips saved. Preview:")
display(df_pooled.head(3))

conn.close()
print("\nAll 3 KPIs were successfully extracted and saved.")

Database connected. Running the KPI Engine...

KPI 1: Cancellations saved. Preview:


,hour_of_day,total_cancellations
0,00,899518
1,01,882669
2,02,1056840



KPI 2: Revenue & Idle Time saved. Preview:


,trip_date,total_revenue_cad,total_idle_minutes
0,2020-01-01,4107247.18,1891020.0
1,2020-01-02,2090023.91,3020548.0
2,2020-01-03,2539215.60,3541700.0



KPI 3: Pooled Trips saved. Preview:


,trip_date,total_pooled_trips,total_trips,pool_efficiency_pct
0,2020-01-01,13407,192453,6.966376
1,2020-01-02,7936,122471,6.479901
2,2020-01-03,8612,150928,5.706032



All 3 KPIs were successfully extracted and saved.
